In [5]:
"""
Module 06: Policy Impact & External Shock Analysis (NYC→Dubai)
================================================================
Data Sources:
  3. Google Trends
  4. Aviation Edge
  + VISA_POLICY_EVENTS from settings

Business Questions:
  1) How did policy/external shocks affect demand signals?
  2) How quickly did demand recover after each shock?
  3) What playbook should OTA use for future disruptions?
"""
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.figsize"] = (12, 6)

print("✅ Setup complete")

✅ Setup complete


In [10]:
from config.settings import (
    VISA_POLICY_EVENTS,
    GOOGLE_CLOUD_API_KEY,
    AVIATION_EDGE_API_KEY,
)

# Google Trends
from src.data_collection.google_trends import (
    generate_synthetic_trends,
)

try:
    from src.data_collection.google_trends import fetch_google_trends
except ImportError:
    fetch_google_trends = None
try:
    from src.data_collection.google_trends import fetch_trends_data
except ImportError:
    fetch_trends_data = None

# Aviation Edge
from src.data_collection.aviation_edge import (
    generate_synthetic_monthly_capacity,
)

# Try real aviation fetch function names (depends on your file)
try:
    from src.data_collection.aviation_edge import fetch_aviation_data
except ImportError:
    fetch_aviation_data = None
try:
    from src.data_collection.aviation_edge import fetch_route_data
except ImportError:
    fetch_route_data = None
try:
    from src.data_collection.aviation_edge import fetch_routes_data
except ImportError:
    fetch_routes_data = None

from src.analysis.policy_impact_analyzer import (
    build_policy_timeline,
    add_policy_regimes,
    pre_post_impact,
    compute_composite_demand_index,
)

print("✅ Imports loaded")

✅ Imports loaded


In [11]:
# --- Trends: real first ---
trends_raw = None
if GOOGLE_CLOUD_API_KEY:
    try:
        if fetch_google_trends is not None:
            trends_raw = fetch_google_trends()
        elif fetch_trends_data is not None:
            trends_raw = fetch_trends_data()
        else:
            raise ImportError("No real trends fetch function found in google_trends.py")

        if trends_raw is None or len(trends_raw) == 0:
            raise ValueError("Empty trends response")

        print(f"✅ Real Google Trends loaded: {trends_raw.shape}")
    except Exception as e:
        print(f"⚠️ Trends fetch failed ({e}) -> using synthetic")
        trends_raw = generate_synthetic_trends()
else:
    print("⚠️ GOOGLE_CLOUD_API_KEY missing -> using synthetic trends")
    trends_raw = generate_synthetic_trends()


# --- Aviation: real first ---
capacity_raw = None
if AVIATION_EDGE_API_KEY:
    try:
        if fetch_aviation_data is not None:
            capacity_raw = fetch_aviation_data()
        elif fetch_routes_data is not None:
            capacity_raw = fetch_routes_data()
        elif fetch_route_data is not None:
            capacity_raw = fetch_route_data()
        else:
            raise ImportError("No real aviation fetch function found in aviation_edge.py")

        if capacity_raw is None or len(capacity_raw) == 0:
            raise ValueError("Empty aviation response")

        print(f"✅ Real Aviation data loaded: {capacity_raw.shape}")
    except Exception as e:
        print(f"⚠️ Aviation fetch failed ({e}) -> using synthetic")
        capacity_raw = generate_synthetic_monthly_capacity()
else:
    print("⚠️ AVIATION_EDGE_API_KEY missing -> using synthetic aviation")
    capacity_raw = generate_synthetic_monthly_capacity()

Fetching Google Trends for 5 keywords...
  Geo: US-NY  |  Timeframe: 2019-01-01 2025-12-31
  Batch 1: ['NYC to Dubai flights', 'Dubai hotels', 'Dubai visa', 'Dubai tourism', 'cheap flights to Dubai']
    ✗ Error: Retry.__init__() got an unexpected keyword argument 'method_whitelist'

⚠️  No data retrieved.
⚠️ Trends fetch failed (Empty trends response) -> using synthetic
Generated synthetic Trends: 365 weeks × 5 keywords
⚠️ Aviation fetch failed (No real aviation fetch function found in aviation_edge.py) -> using synthetic
Generated monthly capacity: 84 months (2019-01 → 2025-12)
